# Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor (Patched for Phase 3/4)\n\nThis notebook contains the complete Phase 2 implementation, mathematical fixes, and verified outputs for the CoD (`hybrid_state_eviction_v13`) pipeline:\n- **Vectorized $\\mathcal{O}(P \cdot d_{\\text{state}} \cdot d_{\\text{head}})$ Readout Decomposition:** Probes terminal Mamba-2 state $S_{P,m}$ via $\\hat{b}_t$ with FP64 self-norm compensation and associativity dust guard ($10^{-6} \\|s\\|_2$).\n- **Matched-Filter Subspace Processing Gain ($g_{\\text{sub}}$):** Restores high-dimensional state-space gain $\\log(d_{\\text{state}} \cdot d_{\\text{head}})$ on coherent natural readouts while strictly preserving Test 3 and Test 4 invariant isolation.\n- **$C^2$-Smooth Monotonic asinh Lower Floor (`_smooth_lower_floor`):** Eliminates the $-10^6$ sentinel cliff and compresses $[-10^6, -20] \\to [-47.6, -20]$, keeping within-age std $\\mathcal{O}(1)$ ($2.18 \\dots 9.16$) for stable Phase 3 $z$-score standardization.\n- **Horizon-Modulated Write-Energy Head Weighting:** Downweights fast local heads at distant horizons $P-t \\gg 64$, allowing long-horizon memory heads to govern token persistence.\n- **Cell 2.0:** Setup, directory paths, environment report, and Phase 1 handover verification.\n- **Cell 2.1 (Patch 2):** Finalized `hooks/survival_metrics.py` with mode-isolated dead-masking (zero log_sir leak into Decay-Only baseline), exact asinh inactive ceiling (-23.28), and device-native calibration.\n- **Cell 2.2:** `hooks/phase2_checks.py` containing Mandatory Test 3 (Single-Write Isolated Limit), Mandatory Test 4 (R5b Equal-Age Collision Separation), and full-model survival audit.\n- **Cell 2.3:** Update `hooks/__init__.py`, purge stale imports, and verify exports.\n- **Cell 2.4:** Smoke test executing Test 3, Test 4, and 36-layer audit on $P=2,048$ saved Phase 1 reference capture.\n- **Cell 2.5 (Re-Run):** Refresh Phase 2 reference feature bundles (`survival_ref_05b_text_2048.pt` and `survival_ref_15b_text_2048.pt`) with patched SurvivalPredictor.\n- **Cell 2.6:** Phase 2 gate check, manifest generation (`checkpoints/phase2/phase2_manifest.json`), and handover summary.

### Cell 2.0 - Setup, Paths & Phase 1 Handover Verification | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor

In [1]:
# ==============================================================================
# Cell 2.0 - Setup, Paths & Phase 1 Handover Verification | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor
# ==============================================================================
import os, sys, json, gc, time, platform
from pathlib import Path
import torch
import transformers

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
CFG_DIR = PROJECT_ROOT / "configs"
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"
PHASE2_DIR = PROJECT_ROOT / "checkpoints" / "phase2"   # FIXED handover path for Phase 3 & Phase 4
PHASE2_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RERUN = False

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Verify Phase 0 and Phase 1 modules and handover artifacts exist
assert (PROJECT_ROOT / "hooks" / "phase0_core.py").exists(), "Missing hooks/phase0_core.py (run Phase 0 first)."
assert (PROJECT_ROOT / "hooks" / "falcon_h1_state_hook.py").exists(), "Missing hooks/falcon_h1_state_hook.py (run Phase 1 first)."
assert (PHASE1_DIR / "phase1_manifest.json").exists(), "Missing checkpoints/phase1/phase1_manifest.json (run Phase 1 first)."

p1_manifest = json.loads((PHASE1_DIR / "phase1_manifest.json").read_text())
assert not p1_manifest.get("problems"), f"Phase 1 manifest reported problems: {p1_manifest.get('problems')}"

for tag in ("05b", "15b"):
    ref_pt = PHASE1_DIR / f"ref_capture_{tag}_text_2048.pt"
    assert ref_pt.exists(), f"Missing Phase 1 reference capture: {ref_pt}"

print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"cuda available: {torch.cuda.is_available()} | gpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
print(f"[Phase 2.0] Verified Phase 1 handover at: {PHASE1_DIR}")
print(f"[Phase 2.0] Initialized Phase 2 handover directory at: {PHASE2_DIR}")

python 3.10.12 | torch 2.4.0 | transformers 4.44.2
cuda available: True | gpus: ['Tesla T4', 'Tesla T4']
[Phase 2.0] Verified Phase 1 handover at: /kaggle/working/hybrid_state_eviction_v13/checkpoints/phase1
[Phase 2.0] Initialized Phase 2 handover directory at: /kaggle/working/hybrid_state_eviction_v13/checkpoints/phase2

### Cell 2.1 (Patch 2) - Fix Dead-Mask Cliff & Feature Leak in hooks/survival_metrics.py | Phase 2 Patch for Phase 3

In [2]:
# ==============================================================================
# Cell 2.1 (Patch 2) - Fix Dead-Mask Cliff & Feature Leak in hooks/survival_metrics.py | Phase 2 Patch for Phase 3
# Fixes: (1) Removes log_sir leak from calibrated_log_decay (Decay-Only baseline)
#        (2) Moves active/inactive dead_mask threshold from -20.5 to exact asinh inactive ceiling (-23.28)
#        (3) Keeps tensor ops on native device (no CPU round-trip per layer)
# ==============================================================================
import sys, importlib
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
SURVIVAL_METRICS_PATH = PROJECT_ROOT / "hooks" / "survival_metrics.py"

SURVIVAL_METRICS_CODE = r'''"""Phase 2 (Patched for Phase 3/4): Realized State-Readout Decomposition & Calibrated Survival Predictor.

Implements Pillar 1 of Master Plan V13:
  1. Vectorized O(P * d_state * d_head_ssm) realized state-readout decomposition:
       r_{t,m}(P) = b_hat_t S_{P,m} = s_{t,m}(P) + nu_{t,m}(P)
  2. Three time-consistent dimensionless survival features per token t, layer l, head m:
       - SIR_{t,l,m}(P) and log_SIR_{t,l,m}(P) (with matched-filter subspace processing gain
         and smooth asinh compression below -20.0 to prevent -1e6 sentinel std explosion)
       - Directional readout cosine alignment c_{t,l,m}(P) in [-1, 1]
       - Cumulative log-survival factor ell_{t,l,m}(P) <= 0
  3. 4-parameter logistic survival calibrator p_hat_surv(t,l,m | P), zero-parameter
     closed-form default rho_surv^closed(t,l,m | P), and horizon-modulated write-weighted
     layer survival p_bar_surv_{t,l}(P).
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

import torch
import torch.nn as nn

EPS_DEFAULT = 1e-8
EPS_REL_DEFAULT = 1e-8
EPS_TINY_FP64 = 1e-30
EPS_FLOOR_FP64 = 1e-300
ASSOC_DUST_REL_TOL = 1e-6
LOG_SIR_SOFT_KINK = -20.0
LOG_SIR_SOFT_SCALE = 2.0
# Exact image of x = -25.0 (the maximum possible inactive_log_sir64 = 2*ell - 25 for ell <= 0)
# under _smooth_lower_floor is -20.0 - 2.0 * asinh(2.5) = -23.294462.
LOG_SIR_INACTIVE_CEIL = -23.28
LOG_SIR_MIN = -60.0
LOG_SIR_EXP_MIN = -60.0
LOG_SIR_MAX = 20.0


def _smooth_lower_floor(x: torch.Tensor, kink: float = LOG_SIR_SOFT_KINK, scale: float = LOG_SIR_SOFT_SCALE) -> torch.Tensor:
    """Smoothly compress values below `kink` via asinh while keeping values >= `kink` exact.
    Strictly monotonic and C^2-continuous everywhere; maps [-1e6, -20] -> [-47.6, -20]."""
    below = kink - scale * torch.asinh(torch.clamp((kink - x) / scale, min=0.0, max=1e12))
    return torch.where(x >= kink, x, below)


@torch.no_grad()
def compute_readout_decomposition(
    b_hat: torch.Tensor,
    u: torch.Tensor,
    alpha: torch.Tensor,
    S_P: torch.Tensor,
    log_alpha: Optional[torch.Tensor] = None,
) -> Dict[str, torch.Tensor]:
    """Compute realized readout r_{t,m}(P), target signal s_{t,m}(P), and interference nu_{t,m}(P)."""
    b64 = b_hat.double()
    u64 = u.double()
    S64 = S_P.double()
    if log_alpha is not None:
        la64 = log_alpha.double()
        a64 = torch.where(la64 >= -85.0, torch.exp(la64), alpha.double())
    else:
        a64 = alpha.double()

    # Vectorized O(P * d_state * d_head_ssm) readout: [B, P, N] x [B, H, N, D] -> [B, P, H, D]
    r64 = torch.einsum("bpn,bhnd->bphd", b64, S64)

    # Exact self-term coefficient ||b_hat_t||_2^2
    b_sq64 = (b64 * b64).sum(dim=-1, keepdim=True).unsqueeze(-1)  # [B, P, 1, 1]
    s64 = b_sq64 * a64.unsqueeze(-1) * u64                        # [B, P, H, D]
    nu64 = r64 - s64                                              # [B, P, H, D]

    # Snap floating-point matrix-multiply associativity dust to exact zero in isolated-write regimes
    s_norm64 = torch.linalg.vector_norm(s64, dim=-1, keepdim=True)
    nu_norm64 = torch.linalg.vector_norm(nu64, dim=-1, keepdim=True)
    dust_mask = (s_norm64 > 0.0) & (nu_norm64 <= (ASSOC_DUST_REL_TOL * s_norm64))
    nu64 = torch.where(dust_mask, torch.zeros_like(nu64), nu64)

    return {
        "r64": r64,
        "s64": s64,
        "nu64": nu64,
        "b_hat_sq_norm64": b_sq64,
    }


@torch.no_grad()
def compute_layer_survival_features(
    layer_record: Dict[str, torch.Tensor],
    eps: float = EPS_DEFAULT,
    eps_rel: float = EPS_REL_DEFAULT,
    return_vectors: bool = False,
    compute_device: Optional[Union[str, torch.device]] = None,
    store_device: Optional[Union[str, torch.device]] = None,
) -> Dict[str, torch.Tensor]:
    """Extract the three time-consistent survival features (SIR, cosine c, log-decay ell) for one layer."""
    orig_dev = layer_record["b_hat"].device
    comp_dev = torch.device(compute_device) if compute_device is not None else orig_dev
    out_dev = torch.device(store_device) if store_device is not None else orig_dev

    b_hat = layer_record["b_hat"].to(comp_dev, non_blocking=True)
    u = layer_record["u"].to(comp_dev, non_blocking=True)
    w = layer_record["w"].to(comp_dev, non_blocking=True)
    log_alpha = layer_record["log_alpha"].to(comp_dev, non_blocking=True)
    alpha = layer_record["alpha"].to(comp_dev, non_blocking=True)
    S_P = layer_record["S_P"].to(comp_dev, non_blocking=True)

    d_state = int(b_hat.shape[-1])
    d_head = int(u.shape[-1])

    decomp = compute_readout_decomposition(b_hat=b_hat, u=u, alpha=alpha, S_P=S_P, log_alpha=log_alpha)
    r64 = decomp["r64"]
    s64 = decomp["s64"]
    nu64 = decomp["nu64"]
    b_sq64 = decomp["b_hat_sq_norm64"].squeeze(-1)  # [B, P, 1]

    r_norm64 = torch.linalg.vector_norm(r64, dim=-1)        # [B, P, H]
    s_norm64 = torch.linalg.vector_norm(s64, dim=-1)        # [B, P, H]
    nu_norm64 = torch.linalg.vector_norm(nu64, dim=-1)      # [B, P, H]
    u_norm64 = torch.linalg.vector_norm(u.double(), dim=-1) # [B, P, H]

    ell64 = log_alpha.double()
    bu_sq64 = b_sq64 * (u_norm64 * u_norm64)
    s_sq64 = s_norm64 * s_norm64
    nu_sq64 = nu_norm64 * nu_norm64

    inactive_mask = (r_norm64 <= 1e-25) | (u_norm64 <= 1e-25) | (bu_sq64 <= (EPS_TINY_FP64 * EPS_TINY_FP64))

    # Scale-invariant unit-vector inner product in FP64
    r_hat64 = r64 / torch.clamp(r_norm64.unsqueeze(-1), min=EPS_TINY_FP64)
    u_hat64 = u.double() / torch.clamp(u_norm64.unsqueeze(-1), min=EPS_TINY_FP64)
    cos64 = torch.clamp((r_hat64 * u_hat64).sum(dim=-1), min=-1.0, max=1.0)
    cos64 = torch.where(inactive_mask, torch.zeros_like(cos64), cos64)

    log_Es64 = 2.0 * ell64 + torch.log(bu_sq64 + EPS_FLOOR_FP64)
    log_nu_sq64 = torch.where(
        nu_sq64 > 0.0,
        torch.log(torch.clamp(nu_sq64, min=EPS_FLOOR_FP64)),
        torch.full_like(nu_sq64, -1e9),
    )
    log_rel_s_sq64 = math.log(float(eps_rel)) + log_Es64
    log_Edenom64 = torch.logaddexp(log_nu_sq64, log_rel_s_sq64)

    subspace_gain_max = math.log(float(max(1, d_state * d_head)))
    interf_frac64 = nu_sq64 / (nu_sq64 + s_sq64 + EPS_TINY_FP64)
    pos_cos64 = torch.clamp(cos64, min=0.0)
    g_sub64 = subspace_gain_max * (pos_cos64 * pos_cos64) * interf_frac64

    raw_log_sir64 = log_Es64 - log_Edenom64 + g_sub64
    inactive_log_sir64 = 2.0 * ell64 - 25.0
    combined_log_sir64 = torch.where(inactive_mask, inactive_log_sir64, raw_log_sir64)
    log_sir64 = torch.clamp(_smooth_lower_floor(combined_log_sir64), min=LOG_SIR_MIN, max=LOG_SIR_MAX)

    sir64 = torch.where(
        inactive_mask,
        torch.zeros_like(log_sir64),
        torch.exp(torch.clamp(log_sir64, min=LOG_SIR_EXP_MIN, max=LOG_SIR_MAX)),
    )

    w_sum = w.sum(dim=-1, keepdim=True) + float(eps)
    alpha_bar = (w * alpha).sum(dim=-1) / w_sum.squeeze(-1)  # [B, P]

    out: Dict[str, torch.Tensor] = {
        "sir": sir64.float().to(out_dev),
        "log_sir": log_sir64.float().to(out_dev),
        "cosine": cos64.float().to(out_dev),
        "log_alpha": log_alpha.float().to(out_dev),
        "alpha": alpha.float().to(out_dev),
        "w": w.float().to(out_dev),
        "r_norm": r_norm64.float().to(out_dev),
        "s_norm": s_norm64.float().to(out_dev),
        "nu_norm": nu_norm64.float().to(out_dev),
        "alpha_bar": alpha_bar.float().to(out_dev),
    }
    if return_vectors:
        out["r"] = r64.float().to(out_dev)
        out["s"] = s64.float().to(out_dev)
        out["nu"] = nu64.float().to(out_dev)
    return out


class SurvivalPredictor(nn.Module):
    """Calibrated & Training-Free State-Survival Predictor for Falcon-H1 Hybrid Models."""

    VALID_MODES = ("closed_form", "calibrated", "calibrated_log_decay")

    def __init__(
        self,
        w1: float = 1.0,
        w2: float = 0.0,
        w3: float = 0.0,
        b0: float = 0.0,
        mode: str = "closed_form",
        eps: float = EPS_DEFAULT,
        eps_rel: float = EPS_REL_DEFAULT,
        feature_mean: Optional[Tuple[float, float, float]] = None,
        feature_std: Optional[Tuple[float, float, float]] = None,
    ):
        super().__init__()
        if mode not in self.VALID_MODES:
            raise ValueError(f"Unsupported mode {mode!r}; expected one of {self.VALID_MODES}.")
        self.mode = mode
        self.eps = float(eps)
        self.eps_rel = float(eps_rel)

        self.register_buffer("weights", torch.tensor([float(w1), float(w2), float(w3)], dtype=torch.float32))
        self.register_buffer("bias", torch.tensor(float(b0), dtype=torch.float32))

        mean_t = torch.zeros(3, dtype=torch.float32) if feature_mean is None else torch.tensor(feature_mean, dtype=torch.float32)
        std_t = torch.ones(3, dtype=torch.float32) if feature_std is None else torch.tensor(feature_std, dtype=torch.float32)
        self.register_buffer("feature_mean", mean_t)
        self.register_buffer("feature_std", torch.clamp(std_t, min=1e-6))
        self.use_standardization = feature_mean is not None or feature_std is not None

    def set_calibrator_weights(
        self,
        w1: float,
        w2: float,
        w3: float,
        b0: float,
        mode: str = "calibrated",
        feature_mean: Optional[Tuple[float, float, float]] = None,
        feature_std: Optional[Tuple[float, float, float]] = None,
    ) -> "SurvivalPredictor":
        """Update calibration parameters in-place (used by Phase 3 L-BFGS Gate 1 fitter)."""
        if mode not in self.VALID_MODES:
            raise ValueError(f"Unsupported mode {mode!r}; expected one of {self.VALID_MODES}.")
        self.mode = mode
        device = self.weights.device
        self.weights.copy_(torch.tensor([float(w1), float(w2), float(w3)], dtype=torch.float32, device=device))
        self.bias.copy_(torch.tensor(float(b0), dtype=torch.float32, device=device))
        if feature_mean is not None:
            self.feature_mean.copy_(torch.tensor(feature_mean, dtype=torch.float32, device=device))
            self.use_standardization = True
        if feature_std is not None:
            self.feature_std.copy_(torch.clamp(torch.tensor(feature_std, dtype=torch.float32, device=device), min=1e-6))
            self.use_standardization = True
        return self

    @torch.no_grad()
    def compute_survival_features(
        self,
        records_or_layer: Dict[Any, Any],
        return_vectors: bool = False,
        compute_device: Optional[Union[str, torch.device]] = None,
        store_device: Optional[Union[str, torch.device]] = None,
    ) -> Dict[Any, Any]:
        """Compute survival features for a single layer record OR a multi-layer {layer_idx: record} dict."""
        if "b_hat" in records_or_layer and torch.is_tensor(records_or_layer["b_hat"]):
            return compute_layer_survival_features(
                records_or_layer,
                eps=self.eps,
                eps_rel=self.eps_rel,
                return_vectors=return_vectors,
                compute_device=compute_device,
                store_device=store_device,
            )
        return {
            int(layer_idx): compute_layer_survival_features(
                rec,
                eps=self.eps,
                eps_rel=self.eps_rel,
                return_vectors=return_vectors,
                compute_device=compute_device,
                store_device=store_device,
            )
            for layer_idx, rec in sorted(records_or_layer.items())
        }

    @torch.no_grad()
    def predict_head_survival_from_features(
        self,
        log_sir: torch.Tensor,
        cosine: torch.Tensor,
        log_alpha: torch.Tensor,
        w: Optional[torch.Tensor] = None,
        mode: Optional[str] = None,
    ) -> torch.Tensor:
        """Compute per-head survival probability p_hat_surv(t,l,m | P) in [0, 1) from extracted features."""
        active_mode = self.mode if mode is None else mode
        if active_mode not in self.VALID_MODES:
            raise ValueError(f"Unsupported mode {active_mode!r}")

        log_sir64 = log_sir.double()
        cos64 = cosine.double()
        ell64 = log_alpha.double()

        # ==============================================================================
        # [MATH-UPGRADE | Phase 2 Patch 2 -> Phase 3]
        # Original Spec: Single dead_mask = (log_sir64 < -20.5) applied across all modes.
        # Upgraded Spec: Mode-isolated inactive masking:
        #                - calibrated_log_decay checks ONLY ell64 <= -85.0 or w <= 1e-30 (zero log_sir leak).
        #                - closed_form and calibrated mask at LOG_SIR_INACTIVE_CEIL (-23.28), the exact
        #                  asinh image of inactive heads (2*ell - 25 <= -25 -> <= -23.294).
        # Reason:        Eliminates artificial -20.5 truncation on active heads and prevents feature
        #                leakage into the Phase 3 Decay-Only ablation baseline.
        # ==============================================================================
        base_dead = ell64 <= -85.0
        if w is not None:
            base_dead = base_dead | (w.double() <= EPS_TINY_FP64)

        if active_mode == "calibrated_log_decay":
            dead_mask = base_dead
        else:
            dead_mask = base_dead | (log_sir64 <= LOG_SIR_INACTIVE_CEIL)

        if active_mode == "closed_form":
            p_head64 = torch.sigmoid(log_sir64)
            p_head64 = torch.where(dead_mask, torch.zeros_like(p_head64), p_head64)
            return p_head64.float()

        dev = log_sir64.device
        wt = self.weights.to(device=dev, dtype=torch.float64)
        b0 = self.bias.to(device=dev, dtype=torch.float64)

        if self.use_standardization:
            fm = self.feature_mean.to(device=dev, dtype=torch.float64)
            fs = self.feature_std.to(device=dev, dtype=torch.float64)
            f_sir = (log_sir64 - fm[0]) / fs[0]
            f_cos = (cos64 - fm[1]) / fs[1]
            f_ell = (ell64 - fm[2]) / fs[2]
        else:
            f_sir, f_cos, f_ell = log_sir64, cos64, ell64

        if active_mode == "calibrated_log_decay":
            logits64 = wt[2] * f_ell + b0
        else:
            logits64 = wt[0] * f_sir + wt[1] * f_cos + wt[2] * f_ell + b0

        p_head64 = torch.sigmoid(logits64)
        p_head64 = torch.where(dead_mask, torch.zeros_like(p_head64), p_head64)
        return p_head64.float()

    @staticmethod
    def aggregate_heads_to_layer(
        p_head: torch.Tensor,
        w: torch.Tensor,
        log_alpha: torch.Tensor,
        eps: float = EPS_DEFAULT,
    ) -> torch.Tensor:
        """Horizon-modulated write-energy aggregation across all H_ssm heads."""
        w64 = w.double()
        ell64 = log_alpha.double()
        horizon_gate64 = 0.02 + 0.98 * torch.exp(torch.clamp(ell64 / 8.0, min=-20.0, max=0.0))
        w_eff64 = w64 * horizon_gate64
        w_denom64 = w_eff64.sum(dim=-1) + float(eps)
        return ((w_eff64 * p_head.double()).sum(dim=-1) / w_denom64).float()

    @torch.no_grad()
    def predict_layer(
        self,
        layer_record_or_features: Dict[str, torch.Tensor],
        mode: Optional[str] = None,
        return_vectors: bool = False,
        compute_device: Optional[Union[str, torch.device]] = None,
        store_device: Optional[Union[str, torch.device]] = None,
    ) -> Dict[str, torch.Tensor]:
        """Compute per-head and horizon-modulated write-weighted layer survival probabilities for a single layer."""
        if "log_sir" in layer_record_or_features and "cosine" in layer_record_or_features:
            feats = dict(layer_record_or_features)
        else:
            feats = compute_layer_survival_features(
                layer_record_or_features,
                eps=self.eps,
                eps_rel=self.eps_rel,
                return_vectors=return_vectors,
                compute_device=compute_device,
                store_device=store_device,
            )

        w = feats["w"]
        la = feats["log_alpha"]
        p_head_closed = self.predict_head_survival_from_features(
            feats["log_sir"], feats["cosine"], la, w=w, mode="closed_form"
        )
        p_head = self.predict_head_survival_from_features(
            feats["log_sir"], feats["cosine"], la, w=w, mode=mode
        )

        p_bar_closed = self.aggregate_heads_to_layer(p_head_closed, w, la, eps=self.eps)
        p_bar = self.aggregate_heads_to_layer(p_head, w, la, eps=self.eps)

        feats["rho_surv_closed_head"] = p_head_closed
        feats["rho_surv_closed_bar"] = p_bar_closed
        feats["p_surv_head"] = p_head
        feats["p_surv_bar"] = p_bar
        feats["deficit_head"] = (1.0 - p_head).clamp(0.0, 1.0)
        feats["deficit_bar"] = (1.0 - p_bar).clamp(0.0, 1.0)
        return feats

    @torch.no_grad()
    def predict_survival(
        self,
        records: Dict[int, Dict[str, torch.Tensor]],
        mode: Optional[str] = None,
        return_vectors: bool = False,
        include_head_tensors: bool = True,
        compute_device: Optional[Union[str, torch.device]] = None,
        store_device: Optional[Union[str, torch.device]] = None,
    ) -> Dict[str, Any]:
        """Run survival prediction across all L layers and return per-layer results + stacked [B, L, P] views."""
        layers_out: Dict[int, Dict[str, torch.Tensor]] = {}
        p_bar_list = []
        def_bar_list = []
        alpha_bar_list = []

        for layer_idx in sorted(records.keys()):
            l_res = self.predict_layer(
                records[layer_idx],
                mode=mode,
                return_vectors=return_vectors,
                compute_device=compute_device,
                store_device=store_device,
            )
            p_bar_list.append(l_res["p_surv_bar"])
            def_bar_list.append(l_res["deficit_bar"])
            alpha_bar_list.append(l_res["alpha_bar"])
            if not include_head_tensors:
                l_res = {
                    k: v
                    for k, v in l_res.items()
                    if k in ("p_surv_bar", "deficit_bar", "alpha_bar", "rho_surv_closed_bar")
                }
            layers_out[int(layer_idx)] = l_res

        p_surv_bar_all = torch.stack(p_bar_list, dim=1)
        deficit_bar_all = torch.stack(def_bar_list, dim=1)
        alpha_bar_all = torch.stack(alpha_bar_list, dim=1)

        return {
            "layers": layers_out,
            "p_surv_bar_all": p_surv_bar_all,
            "deficit_bar_all": deficit_bar_all,
            "alpha_bar_all": alpha_bar_all,
            "mode": self.mode if mode is None else mode,
        }

    def to_config_dict(self) -> Dict[str, Any]:
        w1, w2, w3 = [float(x) for x in self.weights.detach().cpu().tolist()]
        b0 = float(self.bias.detach().cpu().item())
        return {
            "mode": self.mode,
            "w1": w1,
            "w2": w2,
            "w3": w3,
            "b0": b0,
            "eps": self.eps,
            "eps_rel": self.eps_rel,
            "use_standardization": bool(self.use_standardization),
            "feature_mean": [float(x) for x in self.feature_mean.detach().cpu().tolist()],
            "feature_std": [float(x) for x in self.feature_std.detach().cpu().tolist()],
        }

    def save_calibrator(self, path: Union[str, Path], extra_meta: Optional[Dict[str, Any]] = None) -> Path:
        path = Path(path)
        payload = self.to_config_dict()
        if extra_meta:
            payload["meta"] = extra_meta
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2))
        return path

    @classmethod
    def load_calibrator(cls, path: Union[str, Path]) -> "SurvivalPredictor":
        cfg = json.loads(Path(path).read_text())
        use_std = cfg.get("use_standardization", False)
        return cls(
            w1=cfg.get("w1", 1.0),
            w2=cfg.get("w2", 0.0),
            w3=cfg.get("w3", 0.0),
            b0=cfg.get("b0", 0.0),
            mode=cfg.get("mode", "closed_form"),
            eps=cfg.get("eps", EPS_DEFAULT),
            eps_rel=cfg.get("eps_rel", EPS_REL_DEFAULT),
            feature_mean=tuple(cfg["feature_mean"]) if use_std and "feature_mean" in cfg else None,
            feature_std=tuple(cfg["feature_std"]) if use_std and "feature_std" in cfg else None,
        )
'''

SURVIVAL_METRICS_PATH.write_text(SURVIVAL_METRICS_CODE)

for m in [m for m in sys.modules if m == "hooks" or m.startswith("hooks.")]:
    del sys.modules[m]
importlib.invalidate_caches()

print(f"[Cell 2.1 Patch 2] Patched {SURVIVAL_METRICS_PATH} (mode-isolated dead_mask & device-native calibration).")

[Cell 2.1 Patch 2] Patched /kaggle/working/hybrid_state_eviction_v13/hooks/survival_metrics.py (mode-isolated dead_mask & device-native calibration).

### Cell 2.2 - Write hooks/phase2_checks.py (Tests 3, 4 & Full-Model Survival Audit) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor

In [3]:
#Cell 2.2 - Write hooks/phase2_checks.py (Tests 3, 4 & Full-Model Survival Audit) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE2_CHECKS_PATH = PROJECT_ROOT / "hooks" / "phase2_checks.py"

PHASE2_CHECKS_CODE = r'''"""Phase 2 Verification Suite: Test 3 (Single-Write Isolated Limit), Test 4 (R5b Equal-Age Key Collision),
and Full-Model Realized Survival Profile Audit."""

from __future__ import annotations

import time
from typing import Any, Dict, List, Optional, Tuple, Union

import torch
import torch.nn.functional as F

from .survival_metrics import LOG_SIR_MAX, LOG_SIR_MIN, SurvivalPredictor


@torch.no_grad()
def test_03_sir_single_write_limit(
    P: int = 256,
    H_ssm: int = 24,
    d_state: int = 128,
    d_head: int = 64,
    device: Union[str, torch.device] = "cuda:0",
) -> Dict[str, Any]:
    """Mandatory Test 3 (`test_03_sir_single_write_limit`):
    Verify for a single isolated write at t_0 (u_tau = 0 for tau != t_0) that:
      - ||nu_{t_0, m}(P)||_2 == 0
      - c_{t_0, m}(P) == 1.0
      - rho_surv^closed(t_0, m | P) > 1 - 1e-5 for any alpha_{t_0 -> P, m} > 0
        (tested across 12 orders of magnitude of decay, from alpha = 1.0 down to alpha = 1e-12).
    """
    dev = torch.device(device if torch.cuda.is_available() else "cpu")
    g = torch.Generator(device="cpu").manual_seed(42)

    predictor = SurvivalPredictor(mode="closed_form").to(dev)
    probe_positions = [0, 17, P // 2, P - 1]
    head_log_alphas = torch.linspace(-27.631021, 0.0, steps=H_ssm, dtype=torch.float32, device=dev)

    max_nu_norm = 0.0
    min_cosine = 1.0
    min_rho_closed = 1.0
    min_p_surv_bar = 1.0
    per_pos_details: List[Dict[str, Any]] = []

    for t0 in probe_positions:
        B_raw = torch.randn(1, P, d_state, generator=g, dtype=torch.float32).to(dev)
        b_norm = torch.linalg.vector_norm(B_raw, dim=-1)
        b_hat = B_raw / (b_norm.unsqueeze(-1) + 1e-8)

        u = torch.zeros(1, P, H_ssm, d_head, dtype=torch.float32, device=dev)
        u_t0 = torch.randn(1, H_ssm, d_head, generator=g, dtype=torch.float32).to(dev)
        u_t0 = F.normalize(u_t0, p=2, dim=-1) * 2.5
        u[:, t0, :, :] = u_t0
        w = torch.linalg.vector_norm(u, dim=-1)

        log_alpha = torch.zeros(1, P, H_ssm, dtype=torch.float32, device=dev)
        log_alpha[:, t0, :] = head_log_alphas
        alpha = torch.exp(log_alpha.double()).float()

        S_P = torch.einsum("bpn,bphd->bhnd", b_hat.double(), torch.exp(log_alpha.double()).unsqueeze(-1) * u.double()).float()

        rec = {
            "b_hat": b_hat,
            "u": u,
            "w": w,
            "log_alpha": log_alpha,
            "alpha": alpha,
            "S_P": S_P,
        }
        res = predictor.predict_layer(rec, return_vectors=True, compute_device=dev)

        nu_t0 = res["nu_norm"][0, t0]
        cos_t0 = res["cosine"][0, t0]
        rho_t0 = res["rho_surv_closed_head"][0, t0]
        p_bar_t0 = float(res["p_surv_bar"][0, t0].item())

        pos_max_nu = float(nu_t0.max().item())
        pos_min_cos = float(cos_t0.min().item())
        pos_min_rho = float(rho_t0.min().item())

        max_nu_norm = max(max_nu_norm, pos_max_nu)
        min_cosine = min(min_cosine, pos_min_cos)
        min_rho_closed = min(min_rho_closed, pos_min_rho)
        min_p_surv_bar = min(min_p_surv_bar, p_bar_t0)

        per_pos_details.append({
            "t0": t0,
            "max_nu_norm": pos_max_nu,
            "min_cosine": pos_min_cos,
            "min_rho_closed": pos_min_rho,
            "p_surv_bar": p_bar_t0,
        })

    passed = bool(
        max_nu_norm < 1e-6
        and abs(min_cosine - 1.0) < 1e-5
        and min_rho_closed > (1.0 - 1e-5)
        and min_p_surv_bar > (1.0 - 1e-5)
    )
    return {
        "test_name": "test_03_sir_single_write_limit",
        "passed": passed,
        "max_nu_norm": max_nu_norm,
        "min_cosine": min_cosine,
        "min_rho_closed": min_rho_closed,
        "min_p_surv_bar": min_p_surv_bar,
        "alpha_min_tested": float(torch.exp(head_log_alphas.min().double()).item()),
        "probes": per_pos_details,
    }


@torch.no_grad()
def test_04_r5b_equal_age_interference(
    P: int = 256,
    H_ssm: int = 24,
    d_state: int = 128,
    d_head: int = 64,
    n_pairs: int = 16,
    target_collision_cos: float = 0.80,
    device: Union[str, torch.device] = "cuda:0",
    real_layer_record: Optional[Dict[str, torch.Tensor]] = None,
) -> Dict[str, Any]:
    """Mandatory Test 4 (`test_04_r5b_equal_age_interference`):
    Verify on equal-age pairs (P - t_A == P - t_B) where t_A suffers later key collision
    (<b_hat_{t_A}, b_hat_dist> = 0.8) while t_B lies in an orthogonal subspace that:
      - alpha_{t_A -> P, m} == alpha_{t_B -> P, m}
      - SIR_{t_A, m}(P) < 0.2 * SIR_{t_B, m}(P)
      - c_{t_A, m}(P) < c_{t_B, m}(P)
      - p_bar_surv_{t_A}(P) < p_bar_surv_{t_B}(P)
    Also runs a real-backbone R5b injection audit if `real_layer_record` is provided.
    """
    dev = torch.device(device if torch.cuda.is_available() else "cpu")
    g = torch.Generator(device="cpu").manual_seed(1337)
    predictor = SurvivalPredictor(mode="closed_form").to(dev)

    Q, _ = torch.linalg.qr(torch.randn(d_state, d_state, generator=g, dtype=torch.float64))
    Q = Q.to(dev)

    mid = P // 2
    tA_indices = [mid - 32 + 2 * i for i in range(n_pairs)]
    tB_indices = [mid - 32 + 2 * i + 1 for i in range(n_pairs)]
    dist_start = mid + 16
    n_dist_per_pair = 3

    b_hat = torch.zeros(1, P, d_state, dtype=torch.float64, device=dev)
    u = torch.zeros(1, P, H_ssm, d_head, dtype=torch.float64, device=dev)
    log_alpha = torch.zeros(1, P, H_ssm, dtype=torch.float64, device=dev)

    pair_head_log_alpha = torch.linspace(-4.5, -0.15, steps=H_ssm, dtype=torch.float64, device=dev)
    dist_head_log_alpha = pair_head_log_alpha * 0.35

    u_base = F.normalize(torch.randn(n_pairs, H_ssm, d_head, generator=g, dtype=torch.float64).to(dev), p=2, dim=-1) * 1.5
    u_rand = torch.randn(n_pairs, H_ssm, d_head, generator=g, dtype=torch.float64).to(dev)
    proj = (u_rand * u_base).sum(dim=-1, keepdim=True) / ((u_base * u_base).sum(dim=-1, keepdim=True) + 1e-30)
    u_dist_dir = F.normalize(u_rand - proj * u_base, p=2, dim=-1) * 2.2

    cos_val = float(target_collision_cos)
    sin_val = float((1.0 - cos_val * cos_val) ** 0.5)

    for i in range(n_pairs):
        tA = tA_indices[i]
        tB = tB_indices[i]
        eA = Q[3 * i]
        eB = Q[3 * i + 1]
        e_perp = Q[3 * i + 2]

        b_hat[0, tA] = eA
        b_hat[0, tB] = eB
        u[0, tA] = u_base[i]
        u[0, tB] = u_base[i]
        log_alpha[0, tA] = pair_head_log_alpha
        log_alpha[0, tB] = pair_head_log_alpha

        b_dist = cos_val * eA + sin_val * e_perp
        for d_idx in range(n_dist_per_pair):
            tau = dist_start + i * n_dist_per_pair + d_idx
            if tau < P - 4:
                b_hat[0, tau] = b_dist
                u[0, tau] = u_dist_dir[i]
                log_alpha[0, tau] = dist_head_log_alpha

    alpha = torch.exp(log_alpha)
    w = torch.linalg.vector_norm(u, dim=-1)
    S_P = torch.einsum("bpn,bphd->bhnd", b_hat, alpha.unsqueeze(-1) * u)

    rec = {
        "b_hat": b_hat.float(),
        "u": u.float(),
        "w": w.float(),
        "log_alpha": log_alpha.float(),
        "alpha": alpha.float(),
        "S_P": S_P.float(),
    }
    res = predictor.predict_layer(rec, compute_device=dev)

    tA_t = torch.tensor(tA_indices, device=dev, dtype=torch.long)
    tB_t = torch.tensor(tB_indices, device=dev, dtype=torch.long)

    alpha_A = res["alpha"][0, tA_t]
    alpha_B = res["alpha"][0, tB_t]
    sir_A = res["sir"][0, tA_t]
    sir_B = res["sir"][0, tB_t]
    cos_A = res["cosine"][0, tA_t]
    cos_B = res["cosine"][0, tB_t]
    p_bar_A = res["p_surv_bar"][0, tA_t]
    p_bar_B = res["p_surv_bar"][0, tB_t]
    alpha_bar_A = res["alpha_bar"][0, tA_t]
    alpha_bar_B = res["alpha_bar"][0, tB_t]

    max_alpha_diff = float((alpha_A - alpha_B).abs().max().item())
    max_sir_ratio = float((sir_A / (sir_B + 1e-30)).max().item())
    max_cos_diff = float((cos_A - cos_B).max().item())
    mean_p_bar_A = float(p_bar_A.mean().item())
    mean_p_bar_B = float(p_bar_B.mean().item())
    mean_dasc_deficit_diff = float(((1.0 - alpha_bar_A) - (1.0 - alpha_bar_B)).abs().max().item())

    synth_passed = bool(
        max_alpha_diff == 0.0
        and max_sir_ratio < 0.20
        and bool((sir_A < 0.20 * sir_B).all().item())
        and bool((cos_A < cos_B).all().item())
        and bool((p_bar_A < p_bar_B).all().item())
    )

    out: Dict[str, Any] = {
        "test_name": "test_04_r5b_equal_age_interference",
        "passed": synth_passed,
        "n_pairs": n_pairs,
        "target_collision_cos": cos_val,
        "max_alpha_diff_A_vs_B": max_alpha_diff,
        "max_sir_ratio_A_over_B": max_sir_ratio,
        "median_sir_A": float(sir_A.median().item()),
        "median_sir_B": float(sir_B.median().item()),
        "mean_cosine_A": float(cos_A.mean().item()),
        "mean_cosine_B": float(cos_B.mean().item()),
        "max_cos_A_minus_B": max_cos_diff,
        "mean_p_surv_bar_A": mean_p_bar_A,
        "mean_p_surv_bar_B": mean_p_bar_B,
        "dasc_decay_deficit_diff": mean_dasc_deficit_diff,
    }

    # Part B: Real-layer equal-age collision verification using real captured FalconH1Mixer decay profile
    if real_layer_record is not None:
        real_la = real_layer_record["log_alpha"].to(dev)
        P_real = real_la.shape[1]
        log_alpha_real = log_alpha.clone()
        for i in range(n_pairs):
            tA = tA_indices[i]
            tB = tB_indices[i]
            # Match the exact distance-to-P (token age P - tA) in the real captured sequence
            real_pos_pair = max(0, P_real - (P - tA))
            log_alpha_real[0, tA] = real_la[0, real_pos_pair].double()
            log_alpha_real[0, tB] = real_la[0, real_pos_pair].double()
            for d_idx in range(n_dist_per_pair):
                tau = dist_start + i * n_dist_per_pair + d_idx
                if tau < P - 4:
                    real_pos_dist = max(0, P_real - (P - tau))
                    log_alpha_real[0, tau] = real_la[0, real_pos_dist].double()

        alpha_real = torch.exp(log_alpha_real)
        S_P_real = torch.einsum("bpn,bphd->bhnd", b_hat, alpha_real.unsqueeze(-1) * u)
        rec_real = {
            "b_hat": b_hat.float(),
            "u": u.float(),
            "w": w.float(),
            "log_alpha": log_alpha_real.float(),
            "alpha": alpha_real.float(),
            "S_P": S_P_real.float(),
        }
        res_real = predictor.predict_layer(rec_real, compute_device=dev)
        sir_Ar = res_real["sir"][0, tA_t]
        sir_Br = res_real["sir"][0, tB_t]
        cos_Ar = res_real["cosine"][0, tA_t]
        cos_Br = res_real["cosine"][0, tB_t]
        p_bar_Ar = res_real["p_surv_bar"][0, tA_t]
        p_bar_Br = res_real["p_surv_bar"][0, tB_t]

        # Evaluate headwise ratio on surviving heads (where t_B's write has not already vanished by pure decay)
        surviving_head_mask = sir_Br > 1.0
        n_surviving_heads = int(surviving_head_mask[0].sum().item())
        if n_surviving_heads > 0:
            real_ratio = float((sir_Ar[surviving_head_mask] / (sir_Br[surviving_head_mask] + 1e-30)).max().item())
            cos_sep_ok = bool((cos_Ar[surviving_head_mask] < cos_Br[surviving_head_mask]).all().item())
        else:
            real_ratio = 1.0
            cos_sep_ok = False

        mean_pA_real = float(p_bar_Ar.mean().item())
        mean_pB_real = float(p_bar_Br.mean().item())
        real_pass = bool(
            n_surviving_heads > 0
            and real_ratio < 0.20
            and cos_sep_ok
            and bool((p_bar_Ar < p_bar_Br).all().item())
            and mean_pA_real < 0.20 * mean_pB_real
        )
        out["real_decay_profile_check"] = {
            "passed": real_pass,
            "n_surviving_heads_of_24": n_surviving_heads,
            "max_sir_ratio_A_over_B_surviving_heads": real_ratio,
            "mean_p_surv_bar_A": mean_pA_real,
            "mean_p_surv_bar_B": mean_pB_real,
        }
        out["passed"] = bool(out["passed"] and real_pass)

    return out


@torch.no_grad()
def audit_model_survival_profile(
    records: Dict[int, Dict[str, torch.Tensor]],
    predictor: Optional[SurvivalPredictor] = None,
    compute_device: Union[str, torch.device] = "cuda:0",
    store_device: Union[str, torch.device] = "cpu",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Run SurvivalPredictor across all L layers of a captured sequence and verify:
      1. 100% finiteness and strict range invariants on SIR, log_SIR, cosine, log_alpha, p_surv_head, p_surv_bar.
      2. Exact agreement (< 1e-6) between `mode='closed_form'` and `mode='calibrated'` at default weights (1,0,0,0).
      3. Non-trivial within-age variance of log_SIR and cosine at fixed token age (proving survival != recency).
      4. Layerwise expected forgetting deficit profile for Phase 4 risk-adaptive budget allocation.
    """
    dev = torch.device(compute_device if torch.cuda.is_available() else "cpu")
    pred_closed = predictor if predictor is not None else SurvivalPredictor(mode="closed_form")
    pred_closed = pred_closed.to(dev)
    pred_cal_default = SurvivalPredictor(w1=1.0, w2=0.0, w3=0.0, b0=0.0, mode="calibrated").to(dev)

    t0 = time.perf_counter()
    surv_out = pred_closed.predict_survival(
        records,
        return_vectors=False,
        include_head_tensors=True,
        compute_device=dev,
        store_device=store_device,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_s = time.perf_counter() - t0

    layer_summaries: List[Dict[str, Any]] = []
    all_finite = True
    ranges_ok = True
    max_closed_vs_cal_diff = 0.0

    for layer_idx, l_res in sorted(surv_out["layers"].items()):
        sir = l_res["sir"]
        log_sir = l_res["log_sir"]
        cos = l_res["cosine"]
        la = l_res["log_alpha"]
        p_head = l_res["p_surv_head"]
        p_bar = l_res["p_surv_bar"]
        def_bar = l_res["deficit_bar"]
        w = l_res["w"]

        p_cal0 = pred_cal_default.predict_head_survival_from_features(
            log_sir.to(dev), cos.to(dev), la.to(dev), w=w.to(dev), mode="calibrated"
        ).cpu()
        diff_cal = float((p_head.cpu() - p_cal0).abs().max().item())
        max_closed_vs_cal_diff = max(max_closed_vs_cal_diff, diff_cal)

        fin = all(
            bool(torch.isfinite(t).all().item())
            for t in (sir, log_sir, cos, la, p_head, p_bar, def_bar, l_res["nu_norm"], l_res["s_norm"])
        )
        all_finite &= fin

        rng = bool(
            (sir >= 0.0).all().item()
            and (log_sir >= LOG_SIR_MIN - 1.0).all().item()
            and (log_sir <= LOG_SIR_MAX + 1e-4).all().item()
            and (cos >= -1.0 - 1e-5).all().item()
            and (cos <= 1.0 + 1e-5).all().item()
            and (la <= 1e-6).all().item()
            and (p_head >= 0.0).all().item()
            and (p_head <= 1.0 + 1e-6).all().item()
            and (p_bar >= 0.0).all().item()
            and (p_bar <= 1.0 + 1e-6).all().item()
        )
        ranges_ok &= rng

        P = p_bar.shape[1]
        cand_slice = slice(4, max(5, P - 64)) if P > 128 else slice(0, P)
        p_bar_cand = p_bar[0, cand_slice]
        def_bar_cand = def_bar[0, cand_slice]
        cos_cand = cos[0, cand_slice]
        log_sir_cand = log_sir[0, cand_slice]

        mid = P // 2
        age_band = slice(max(0, mid - 16), min(P, mid + 16))
        within_age_log_sir_std = float(log_sir[0, age_band].std().item())
        within_age_cos_std = float(cos[0, age_band].std().item())

        layer_summaries.append({
            "layer": int(layer_idx),
            "finite": fin,
            "ranges_ok": rng,
            "p_surv_bar_mean_cand": round(float(p_bar_cand.mean().item()), 5),
            "p_surv_bar_std_cand": round(float(p_bar_cand.std().item()), 5),
            "deficit_bar_mean_cand": round(float(def_bar_cand.mean().item()), 5),
            "log_sir_mean_cand": round(float(log_sir_cand.mean().item()), 4),
            "log_sir_max_head_mean_cand": round(float(log_sir_cand.max(dim=-1).values.mean().item()), 4),
            "cosine_mean_cand": round(float(cos_cand.mean().item()), 4),
            "terminal_token_p_surv_bar": round(float(p_bar[0, -1].item()), 5),
            "within_age_log_sir_std": round(within_age_log_sir_std, 4),
            "within_age_cos_std": round(within_age_cos_std, 4),
        })

    deficits_by_layer = [r["deficit_bar_mean_cand"] for r in layer_summaries]
    within_age_stds = [r["within_age_log_sir_std"] for r in layer_summaries]
    within_age_cos_stds = [r["within_age_cos_std"] for r in layer_summaries]
    P_seq = int(surv_out["p_surv_bar_all"].shape[-1])
    L_layers = len(layer_summaries)

    summary = {
        "n_layers": L_layers,
        "P": P_seq,
        "all_finite": bool(all_finite),
        "ranges_ok": bool(ranges_ok),
        "max_closed_vs_cal_default_diff": max_closed_vs_cal_diff,
        "mean_candidate_p_surv_bar": round(float(sum(r["p_surv_bar_mean_cand"] for r in layer_summaries) / max(1, L_layers)), 5),
        "mean_candidate_deficit_bar": round(float(sum(deficits_by_layer) / max(1, L_layers)), 5),
        "min_layer_deficit_bar": round(min(deficits_by_layer), 5),
        "max_layer_deficit_bar": round(max(deficits_by_layer), 5),
        "min_within_age_log_sir_std": round(min(within_age_stds), 4),
        "mean_within_age_log_sir_std": round(float(sum(within_age_stds) / max(1, L_layers)), 4),
        "min_within_age_cos_std": round(min(within_age_cos_stds), 4),
        "survival_compute_seconds": round(elapsed_s, 4),
        "audit_pass": bool(
            all_finite
            and ranges_ok
            and max_closed_vs_cal_diff < 1e-6
            and min(within_age_stds) > 0.05
            and min(within_age_cos_stds) > 0.01
        ),
    }
    return {"summary": summary, "layers": layer_summaries}, surv_out
'''

PHASE2_CHECKS_PATH.write_text(PHASE2_CHECKS_CODE)
print(f"[Phase 2.2] Wrote updated {PHASE2_CHECKS_PATH}")

[Phase 2.2] Wrote updated /kaggle/working/hybrid_state_eviction_v13/hooks/phase2_checks.py

### Cell 2.3 - Update hooks/__init__.py, Purge Stale Imports & Verify Signatures | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor

In [4]:
#Cell 2.3 - Update hooks/__init__.py, Purge Stale Imports & Verify Signatures | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor
import sys, importlib
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")

for m in [m for m in sys.modules if m == "hooks" or m.startswith("hooks.")]:
    del sys.modules[m]
importlib.invalidate_caches()

from hooks import (
    SurvivalPredictor,
    compute_readout_decomposition,
    compute_layer_survival_features,
    test_03_sir_single_write_limit,
    test_04_r5b_equal_age_interference,
    audit_model_survival_profile,
)

print("[Phase 2.3] Reloaded updated hooks package and verified all Phase 2 exports.")

[Phase 2.3] Reloaded updated hooks package and verified all Phase 2 exports.

### Cell 2.4 - SMOKE TEST (Test 3, Test 4 + 0.5B Saved Capture P=2048) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor

In [5]:
#Cell 2.4 - SMOKE TEST (Test 3, Test 4 + 0.5B Saved Capture P=2048) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor
import gc, json, torch
from pathlib import Path
from hooks import (
    SurvivalPredictor,
    test_03_sir_single_write_limit,
    test_04_r5b_equal_age_interference,
    audit_model_survival_profile,
)

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"
PHASE2_DIR = PROJECT_ROOT / "checkpoints" / "phase2"

t3_05b = test_03_sir_single_write_limit(P=256, H_ssm=24, d_state=128, d_head=64, device="cuda:0")
t3_15b = test_03_sir_single_write_limit(P=256, H_ssm=24, d_state=256, d_head=64, device="cuda:0")
print("=== Test 3 (Single-Write Isolated Limit) ===")
print("  d_state=128 (0.5B)    :", json.dumps({k: v for k, v in t3_05b.items() if k != "probes"}))
print("  d_state=256 (1.5B)    :", json.dumps({k: v for k, v in t3_15b.items() if k != "probes"}))

ref_05b = torch.load(PHASE1_DIR / "ref_capture_05b_text_2048.pt", map_location="cpu")
layers_05b = ref_05b["layers"]

t4_05b = test_04_r5b_equal_age_interference(
    P=256, H_ssm=24, d_state=128, d_head=64, n_pairs=16, target_collision_cos=0.80,
    device="cuda:0", real_layer_record=layers_05b[18],
)
print("\n=== Test 4 (R5b Equal-Age Key-Collision Separation) ===")
print(json.dumps(t4_05b, indent=2))

audit_rep, surv_out = audit_model_survival_profile(layers_05b, compute_device="cuda:0", store_device="cpu")
print("\n=== 0.5B P=2048 Survival Profile Audit Summary ===")
print(json.dumps(audit_rep["summary"], indent=2))
print("Sample layers (L0, L18, L35):")
for idx in (0, 18, 35):
    print(" ", audit_rep["layers"][idx])

smoke_ok = bool(t3_05b["passed"] and t3_15b["passed"] and t4_05b["passed"] and audit_rep["summary"]["audit_pass"])
(PHASE2_DIR / "smoke_report_phase2.json").write_text(json.dumps({
    "test3_d128": t3_05b,
    "test3_d256": t3_15b,
    "test4_05b": t4_05b,
    "audit_05b_2048": audit_rep["summary"],
    "smoke_pass": smoke_ok,
}, indent=2))

del ref_05b, layers_05b, surv_out
gc.collect()
torch.cuda.empty_cache()

print("\n" + ("PHASE 2 SMOKE PASS" if smoke_ok else "PHASE 2 SMOKE FAIL"))
if not smoke_ok:
    raise SystemExit("Phase 2 Smoke Test failed -> stop and inspect output before running Cell 2.5.")

=== Test 3 (Single-Write Isolated Limit) ===
  d_state=128 (0.5B)    : {"test_name": "test_03_sir_single_write_limit", "passed": true, "max_nu_norm": 0.0, "min_cosine": 1.0, "min_rho_closed": 1.0, "min_p_surv_bar": 1.0, "alpha_min_tested": 1e-12}
  d_state=256 (1.5B)    : {"test_name": "test_03_sir_single_write_limit", "passed": true, "max_nu_norm": 0.0, "min_cosine": 1.0, "min_rho_closed": 1.0, "min_p_surv_bar": 1.0, "alpha_min_tested": 1e-12}

=== Test 4 (R5b Equal-Age Key-Collision Separation) ===
{
  "test_name": "test_04_r5b_equal_age_interference",
  "passed": true,
  "n_pairs": 16,
  "target_collision_cos": 0.8,
  "max_alpha_diff_A_vs_B": 0.0,
  "max_sir_ratio_A_over_B": 6.64e-10,
  "median_sir_A": 0.01235,
  "median_sir_B": 18589214.0,
  "mean_cosine_A": 0.0122,
  "mean_cosine_B": 1.0,
  "max_cos_A_minus_B": -0.9878,
  "mean_p_surv_bar_A": 0.0122,
  "mean_p_surv_bar_B": 1.0,
  "dasc_decay_deficit_diff": 0.0,
  "real_decay_profile_check": {
    "passed": true,
    "n_surviving_h

### Cell 2.5 (Re-Run) - Refresh Phase 2 Reference Bundles (0.5B & 1.5B-Deep) | Phase 2 Patch Verification

In [6]:
# ==============================================================================
# Cell 2.5 (Re-Run) - Refresh Phase 2 Reference Bundles (0.5B & 1.5B-Deep) | Phase 2 Patch Verification
# Recomputes survival_ref_{tag}_text_2048.pt and report_phase2_{tag}.json with the patched SurvivalPredictor.
# ==============================================================================
import gc, json, torch
from pathlib import Path
from hooks import (
    SurvivalPredictor,
    test_03_sir_single_write_limit,
    test_04_r5b_equal_age_interference,
    audit_model_survival_profile,
)

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE1_DIR = PROJECT_ROOT / "checkpoints" / "phase1"
PHASE2_DIR = PROJECT_ROOT / "checkpoints" / "phase2"

for tag, d_state in (("05b", 128), ("15b", 256)):
    predictor = SurvivalPredictor(mode="closed_form")
    t3 = test_03_sir_single_write_limit(P=256, H_ssm=24, d_state=d_state, d_head=64, device="cuda:0")

    ref_2048_blob = torch.load(PHASE1_DIR / f"ref_capture_{tag}_text_2048.pt", map_location="cpu")
    records_2048 = ref_2048_blob["layers"]
    L = len(records_2048)

    t4 = test_04_r5b_equal_age_interference(
        P=256, H_ssm=24, d_state=d_state, d_head=64, n_pairs=16, target_collision_cos=0.80,
        device="cuda:0", real_layer_record=records_2048[L // 2],
    )
    audit_2048, surv_2048 = audit_model_survival_profile(
        records_2048, predictor=predictor, compute_device="cuda:0", store_device="cpu"
    )

    compact_layers_2048 = {
        int(l_idx): {
            "log_sir": l_dict["log_sir"].cpu(),
            "cosine": l_dict["cosine"].cpu(),
            "log_alpha": l_dict["log_alpha"].cpu(),
            "w": l_dict["w"].cpu(),
            "p_surv_head": l_dict["p_surv_head"].cpu(),
            "p_surv_bar": l_dict["p_surv_bar"].cpu(),
            "deficit_bar": l_dict["deficit_bar"].cpu(),
            "alpha_bar": l_dict["alpha_bar"].cpu(),
        }
        for l_idx, l_dict in surv_2048["layers"].items()
    }
    torch.save({
        "meta": {
            "tag": tag,
            "model": ref_2048_blob["meta"]["model"],
            "L": L,
            "P": 2048,
            "d_state": d_state,
            "calibrator_config": predictor.to_config_dict(),
        },
        "input_ids": ref_2048_blob["input_ids"].cpu(),
        "p_surv_bar_all": surv_2048["p_surv_bar_all"].cpu(),
        "deficit_bar_all": surv_2048["deficit_bar_all"].cpu(),
        "alpha_bar_all": surv_2048["alpha_bar_all"].cpu(),
        "layers": compact_layers_2048,
    }, PHASE2_DIR / f"survival_ref_{tag}_text_2048.pt")

    s = audit_2048["summary"]
    print(f"[Patched Phase 2 | {tag}] test3={t3['passed']} | test4={t4['passed']} | "
          f"mean_p_surv={s['mean_candidate_p_surv_bar']:.4f} | "
          f"deficit_range=[{s['min_layer_deficit_bar']:.4f}, {s['max_layer_deficit_bar']:.4f}] | "
          f"within_age_log_sir_std=[{s['min_within_age_log_sir_std']:.3f}, {s['mean_within_age_log_sir_std']:.3f}]")

    # Update saved report_phase2_{tag}.json with refreshed P=2048 metrics
    rep_path = PHASE2_DIR / f"report_phase2_{tag}.json"
    if rep_path.exists():
        rep = json.loads(rep_path.read_text())
        rep["test3"] = t3
        rep["test4"] = t4
        rep["runs"]["text_2048"] = audit_2048
        rep_path.write_text(json.dumps(rep, indent=2))

    del ref_2048_blob, records_2048, surv_2048, compact_layers_2048
    gc.collect()
    torch.cuda.empty_cache()

[Patched Phase 2 | 05b] test3=True | test4=True | mean_p_surv=0.0037 | deficit_range=[0.9749, 1.0000] | within_age_log_sir_std=[2.182, 7.531]
[Patched Phase 2 | 15b] test3=True | test4=True | mean_p_surv=0.0178 | deficit_range=[0.9291, 0.9993] | within_age_log_sir_std=[4.434, 9.160]

### Cell 2.6 - PHASE 2 CHECK CELL (Validates All Gates, Handover Files & Prints Summary) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor

In [7]:
#Cell 2.6 - PHASE 2 CHECK CELL (Validates All Gates, Handover Files & Prints Summary) | Phase 2: Realized State-Readout Decomposition & Calibrated Survival Predictor
import json, torch, transformers
from pathlib import Path
from hooks import SurvivalPredictor

PROJECT_ROOT = Path("/kaggle/working/hybrid_state_eviction_v13")
PHASE2_DIR = PROJECT_ROOT / "checkpoints" / "phase2"

problems = []
summary_out = {}

for tag, L_expected, dstate in (("05b", 36, 128), ("15b", 66, 256)):
    rep_file = PHASE2_DIR / f"report_phase2_{tag}.json"
    ref_file = PHASE2_DIR / f"survival_ref_{tag}_text_2048.pt"
    cal_file = PHASE2_DIR / f"default_calibrator_{tag}.json"

    for f in (rep_file, ref_file, cal_file):
        if not f.exists():
            problems.append(f"missing {f.name}")
    if not rep_file.exists():
        continue

    rep = json.loads(rep_file.read_text())
    t3 = rep["test3"]
    t4 = rep["test4"]

    if not t3["passed"]:
        problems.append(f"{tag}: Test 3 FAIL (max_nu={t3['max_nu_norm']}, min_cos={t3['min_cosine']}, min_rho={t3['min_rho_closed']})")
    if not t4["passed"]:
        problems.append(f"{tag}: Test 4 FAIL (max_sir_ratio={t4['max_sir_ratio_A_over_B']}, alpha_diff={t4['max_alpha_diff_A_vs_B']})")

    # Verify calibrator serialization round-trip
    loaded_pred = SurvivalPredictor.load_calibrator(cal_file)
    if loaded_pred.mode != "closed_form":
        problems.append(f"{tag}: unexpected default calibrator mode {loaded_pred.mode}")

    # Verify saved survival reference bundle shapes & dtypes
    if ref_file.exists():
        blob = torch.load(ref_file, map_location="cpu")
        if tuple(blob["p_surv_bar_all"].shape) != (1, L_expected, 2048):
            problems.append(f"{tag}: p_surv_bar_all shape {tuple(blob['p_surv_bar_all'].shape)} != (1, {L_expected}, 2048)")
        if tuple(blob["deficit_bar_all"].shape) != (1, L_expected, 2048):
            problems.append(f"{tag}: deficit_bar_all shape mismatch")
        l0 = blob["layers"][0]
        for k, exp_shp in (
            ("log_sir", (1, 2048, 24)),
            ("cosine", (1, 2048, 24)),
            ("log_alpha", (1, 2048, 24)),
            ("w", (1, 2048, 24)),
            ("p_surv_head", (1, 2048, 24)),
            ("p_surv_bar", (1, 2048)),
            ("deficit_bar", (1, 2048)),
            ("alpha_bar", (1, 2048)),
        ):
            if tuple(l0[k].shape) != exp_shp:
                problems.append(f"{tag}: layer0[{k}] shape {tuple(l0[k].shape)} != {exp_shp}")
            if l0[k].dtype != torch.float32:
                problems.append(f"{tag}: layer0[{k}] dtype {l0[k].dtype} != float32")
        del blob

    for run_name, r_data in rep["runs"].items():
        s = r_data["summary"]
        if not s["audit_pass"]:
            problems.append(f"{tag}/{run_name}: survival audit FAIL (finite={s['all_finite']}, ranges={s['ranges_ok']})")

    summary_out[tag] = {
        "L": rep["L"],
        "d_state": rep["d_state"],
        "test3_pass": t3["passed"],
        "test3_max_nu_norm": t3["max_nu_norm"],
        "test3_min_cosine": t3["min_cosine"],
        "test3_min_rho_closed": t3["min_rho_closed"],
        "test4_pass": t4["passed"],
        "test4_max_alpha_diff": t4["max_alpha_diff_A_vs_B"],
        "test4_max_sir_ratio_A_over_B": round(t4["max_sir_ratio_A_over_B"], 6),
        "test4_mean_cosine_A_vs_B": [round(t4["mean_cosine_A"], 4), round(t4["mean_cosine_B"], 4)],
        "test4_mean_p_surv_bar_A_vs_B": [round(t4["mean_p_surv_bar_A"], 4), round(t4["mean_p_surv_bar_B"], 4)],
        "text_2048": rep["runs"]["text_2048"]["summary"],
        "text_8192": rep["runs"]["text_8192"]["summary"],
        "phase2_pass": rep["phase2_pass"],
    }

manifest = {
    "phase": 2,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "handover_dir": str(PHASE2_DIR),
    "files": sorted(p.name for p in PHASE2_DIR.iterdir()),
    "survival_module": "hooks/survival_metrics.py",
    "checks_module": "hooks/phase2_checks.py",
    "problems": problems,
}
(PHASE2_DIR / "phase2_manifest.json").write_text(json.dumps(manifest, indent=2))

print("=" * 72)
print("PHASE 2 SUMMARY (paste everything between the lines back to me)")
print("=" * 72)
print(json.dumps(summary_out, indent=2))
print("files:", manifest["files"])
print("problems:", problems if problems else "NONE")
print("=" * 72)
print("PHASE 2 GATE:", "PASS" if not problems else "FAIL")

PHASE 2 SUMMARY (paste everything between the lines back to me)
{
  "05b": {
    "L": 36,
    "d_state": 128,
    "test3_pass": true,
    "test3_max_nu_norm": 0.0,
    "test3_min_cosine": 1.0,
    "test3_min_rho_closed": 1.0,
    "test4_pass": true,
    "test4_max_alpha_diff": 0.0,
    "test4_max_sir_ratio_A_over_B": 6.64e-10,
    "test4_mean_cosine_A_vs_B": [
      0.0122,
      1.0
    ],
    "test4_mean_p_surv_bar_A_vs_B": [
      0.0122,
      1.0
    ],
    "text_2048": {
      "n_layers": 36,
      "P": 2048,
      "all_finite": true,
      "ranges_ok": true,
      "max_closed_vs_cal_default_diff": 0.0,
      "mean_candidate_p_surv_bar": 0.0037,
      "mean_candidate_deficit_bar": 0.9963,
      "min_layer_deficit_bar": 0.9749,
      "max_layer_deficit_bar": 1.0000,
      "min_within_age_log_sir_std": 2.182,
      "mean_within_age_log_sir_std": 7.531,
      "min_within_age_cos_std": 0.0824,
      "survival_compute_seconds": 0.038,
      "audit_pass": true
    },
    "phase2_pass":